# 09 — Refresh verified source before launching a new Spark application
This uploads source/config only, keeps the session run ID and creates no compute resources. Do not change the model design after the test freeze.

In [ ]:
from pathlib import Path
import json, subprocess
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks": REPO_ROOT = REPO_ROOT.parent
previous = json.loads((REPO_ROOT / "artifacts/execution/runtime.json").read_text())
PROJECT_ID, BUCKET, RUN_ID = previous["project_id"], previous["bucket"], previous["run_id"]
BILLING_ACCOUNT_ID, ALLOWED_CREDIT_ID = "01DAD8-AE5F1A-E3910A", "3LU63BPGY77JFDQQ"
if previous["billing_account_id"] != BILLING_ACCOUNT_ID or previous["credit_id"] != ALLOWED_CREDIT_ID:
    raise RuntimeError("Unapproved runtime account/credit.")
def run_gcloud(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True, check=True)
    return result.stdout
billing = json.loads(run_gcloud(["billing", "projects", "describe", PROJECT_ID, "--format=json"]))
if not billing.get("billingEnabled") or billing.get("billingAccountName") != "billingAccounts/" + BILLING_ACCOUNT_ID:
    raise RuntimeError("Education linkage changed.")
# This refresh uploads code only; it creates no additional compute resources.
PLANNED_SESSION_COST_USD = 8.0
def require_cost_gate(estimate):
    if estimate != 8.0: raise RuntimeError("Unexpected session allocation.")


In [ ]:
# Bundle only explicit project code/configuration; exclude secrets and raw data.
import hashlib
import sys
import zipfile
sys.path.insert(0, str(REPO_ROOT / "src"))
from rainfall.config import load_config, config_hash
RUN_ID = previous["run_id"]
artifact_dir = REPO_ROOT / "artifacts" / "execution"
artifact_dir.mkdir(parents=True, exist_ok=True)
bundle = artifact_dir / "source_bundle.zip"
files_to_bundle = list((REPO_ROOT / "src" / "rainfall").glob("*.py"))
files_to_bundle += list((REPO_ROOT / "scripts").glob("*.py"))
files_to_bundle += [REPO_ROOT / "config" / "experiment.json", REPO_ROOT / "manifests" / "input_manifest.json", REPO_ROOT / "requirements-cluster.txt"]
files_to_bundle += [path for path in (REPO_ROOT / "notebooks").glob("*.ipynb") if path.name[:2] in {"01", "02", "03", "04", "05"}]
wheel = REPO_ROOT / "artifacts" / "wheels" / "scienceplots-2.2.2-py3-none-any.whl"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(files_to_bundle):
        archive.write(path, str(path.relative_to(REPO_ROOT)))
    archive.write(wheel, f"wheels/{wheel.name}")
config = load_config(REPO_ROOT / "config" / "experiment.json")
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
runtime = {"project_id": PROJECT_ID, "bucket": BUCKET, "run_id": RUN_ID, "billing_account_id": BILLING_ACCOUNT_ID, "credit_id": ALLOWED_CREDIT_ID, "config_sha256": config_hash(config), "source_bundle_sha256": hashlib.sha256(bundle.read_bytes()).hexdigest(), "git_commit": commit}
runtime_path = artifact_dir / "runtime.json"
runtime_path.write_text(json.dumps(runtime, indent=2) + "\n")
require_cost_gate(PLANNED_SESSION_COST_USD)
run_gcloud(["storage", "cp", str(bundle), f"gs://{BUCKET}/code/source_bundle.zip"])
run_gcloud(["storage", "cp", str(runtime_path), f"gs://{BUCKET}/config/runtime.json"])
for path in files_to_bundle:
    if path.suffix == ".ipynb":
        run_gcloud(["storage", "cp", str(path), f"gs://{BUCKET}/notebooks/source/{path.name}"])
display(runtime)
